In [1]:
from __future__ import annotations

import numpy
import random
import pandas
import time
import os
from deap import creator
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef
from scipy.stats import pointbiserialr

from training_config import TrainingConfig
from training_utils import best_sign_consistency_index, knee_point_index
from multi_objective_training import MultiObjectiveTraining
from single_objective_training import SingleObjectiveTraining
from evaluation_utils import (
    get_continuous_columns, get_dummy_columns,
    build_model_package, evaluate_and_save,
    compute_stability_matrix, plot_stability_heatmap, plot_feature_frequency,
    compute_vif_across_seeds, plot_vif_boxplot,
)

In [2]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    numpy.random.seed(seed)

In [3]:
#CSV_TRAIN_PATH: str = "readmit/readmit_130_hospitals_preprocessed_train_data.csv"
#CSV_TEST_PATH: str = "readmit/readmit_130_hospitals_preprocessed_test_data.csv"
#TARGET_COLUMN: str = "target_readmitted"
#CSV_TRAIN_PATH: str = "arrhythmia/arrhythmia_preprocessed_train_data.csv"
#CSV_TEST_PATH: str = "arrhythmia/arrhythmia_preprocessed_test_data.csv"
#TARGET_COLUMN: str = "Outcome"
CSV_TRAIN_PATH: str = "D:\\RoutePatch\\GitWorking\\multi-objective-regression\\test_data\\rad_fusion_train_modified.csv"
CSV_VALIDATION_PATH: str = "D:\\RoutePatch\\GitWorking\\multi-objective-regression\\test_data\\rad_fusion_validation_modified.csv"
CSV_TEST_PATH: str = "D:\\RoutePatch\\GitWorking\\multi-objective-regression\\test_data\\rad_fusion_test_modified.csv"
TARGET_COLUMN: str = "label"

RESULT_PATH: str = time.strftime("%Y-%m-%d_%H-%M-%S")
RESULT_PATH_MULTI: str = f"{RESULT_PATH}\\multi"
RESULT_PATH_SINGLE: str = f"{RESULT_PATH}\\single"
RESULT_PATH_EVAL: str = f"{RESULT_PATH}\\evaluation"

USE_KNEE_POINT_SELECTION: bool = True

In [4]:
# Load train set
files: list[str] = [CSV_TRAIN_PATH, CSV_VALIDATION_PATH]
df_list: list[pandas.DataFrame] = [pandas.read_csv(file) for file in files]
df_train: pandas.DataFrame = pandas.concat(df_list, ignore_index=True)
df_train[TARGET_COLUMN] = df_train[TARGET_COLUMN].astype(int)
df_train: pandas.DataFrame = df_train[[    "label",
                                           "Female",
                                           "current_age_yrs",
                                           "SMOKER_Y",
                                           "COMPLICATIONS MAINLY RELATED TO PREGNANCY:frequency",
                                           " COMPLICATIONS OCCURRING MAINLY IN THE COURSE OF LABOR AND DELIVERY:frequency",
                                           "COMPLICATIONS OF THE PUERPERIUM:frequency",
                                           "NORMAL DELIVERY, AND OTHER INDICATIONS FOR CARE IN PREGNANCY, LABOR, AND DELIVERY:frequency",
                                           " Bulbus cordis anomalies and anomalies of cardiac septal closure:frequency",
                                           "Congenital anomalies of urinary system:frequency",
                                           " Other and unspecified congenital anomalies:frequency",
                                           "Other congenital anomalies of circulatory system:frequency",
                                           " Other congenital anomalies of heart:frequency",
                                           "Other congenital anomalies of nervous system:frequency",
                                           " Other congenital musculoskeletal anomalies:frequency",
                                           "Acquired hemolytic anemias:frequency",
                                           "Aplastic anemia and other bone marrow failure syndromes:frequency",
                                           " Coagulation defects:frequency",
                                           "Diseases of white blood cells:frequency",
                                           "Iron deficiency anemias:frequency",
                                           "Other and unspecified anemias:frequency",
                                           "Other diseases of blood and blood-forming organs:frequency",
                                           "Purpura and other hemorrhagic conditions:frequency",
                                           "CEREBROVASCULAR DISEASE:frequency",
                                           "CHRONIC RHEUMATIC HEART DISEASE:frequency",
                                           "DISEASES OF ARTERIES, ARTERIOLES, AND CAPILLARIES:frequency",
                                           "DISEASES OF PULMONARY CIRCULATION:frequency",
                                           "DISEASES OF VEINS AND LYMPHATICS, AND OTHER DISEASES OF CIRCULATORY SYSTEM:frequency",
                                           " HYPERTENSIVE DISEASE:frequency",
                                           "ISCHEMIC HEART DISEASE:frequency",
                                           " OTHER FORMS OF HEART DISEASE:frequency",
                                           " APPENDICITIS:frequency",
                                           "DISEASES OF ESOPHAGUS, STOMACH, AND DUODENUM:frequency",
                                           "DISEASES OF ORAL CAVITY, SALIVARY GLANDS, AND JAWS:frequency",
                                           "HERNIA OF ABDOMINAL CAVITY:frequency",
                                           " NONINFECTIOUS ENTERITIS AND COLITIS:frequency",
                                           " OTHER DISEASES OF DIGESTIVE SYSTEM:frequency",
                                           "OTHER DISEASES OF INTESTINES AND PERITONEUM:frequency",
                                           "DISEASES OF MALE GENITAL ORGANS:frequency",
                                           " DISORDERS OF BREAST:frequency",
                                           "INFLAMMATORY DISEASE OF FEMALE PELVIC ORGANS:frequency",
                                           " NEPHRITIS, NEPHROTIC SYNDROME, AND NEPHROSIS:frequency",
                                           " OTHER DISEASES OF URINARY SYSTEM:frequency",
                                           "OTHER DISORDERS OF FEMALE GENITAL TRACT:frequency",
                                           " ARTHROPATHIES AND RELATED DISORDERS:frequency",
                                           "DORSOPATHIES:frequency",
                                           "OSTEOPATHIES, CHONDROPATHIES, AND ACQUIRED MUSCULOSKELETAL DEFORMITIES:frequency",
                                           "RHEUMATISM, EXCLUDING THE BACK:frequency",
                                           "DISEASES OF THE EAR AND MASTOID PROCESS:frequency",
                                           "DISORDERS OF THE EYE AND ADNEXA:frequency",
                                           "DISORDERS OF THE PERIPHERAL NERVOUS SYSTEM:frequency",
                                           "HEREDITARY AND DEGENERATIVE DISEASES OF THE CENTRAL NERVOUS SYSTEM:frequency",
                                           " INFLAMMATORY DISEASES OF THE CENTRAL NERVOUS SYSTEM:frequency",
                                           "ORGANIC SLEEP DISORDERS:frequency",
                                           "OTHER DISORDERS OF THE CENTRAL NERVOUS SYSTEM:frequency",
                                           " PAIN:frequency",
                                           "ACUTE RESPIRATORY INFECTIONS:frequency",
                                           "CHRONIC OBSTRUCTIVE PULMONARY DISEASE AND ALLIED CONDITIONS:frequency",
                                           "OTHER DISEASES OF RESPIRATORY SYSTEM:frequency",
                                           "OTHER DISEASES OF THE UPPER RESPIRATORY TRACT:frequency",
                                           "PNEUMOCONIOSES AND OTHER LUNG DISEASES DUE TO EXTERNAL AGENTS:frequency",
                                           " PNEUMONIA AND INFLUENZA:frequency",
                                           "INFECTIONS OF SKIN AND SUBCUTANEOUS TISSUE:frequency",
                                           "OTHER DISEASES OF SKIN AND SUBCUTANEOUS TISSUE:frequency",
                                           "OTHER INFLAMMATORY CONDITIONS OF SKIN AND SUBCUTANEOUS TISSUE:frequency",
                                           " DISEASES OF OTHER ENDOCRINE GLANDS:frequency",
                                           "DISORDERS OF THYROID GLAND:frequency",
                                           " NUTRITIONAL DEFICIENCIES:frequency",
                                           "OTHER METABOLIC AND IMMUNITY DISORDERS:frequency",
                                           " INTESTINAL INFECTIOUS DISEASES:frequency",
                                           "MYCOSES:frequency",
                                           " OTHER BACTERIAL DISEASES:frequency",
                                           "OTHER DISEASES DUE TO VIRUSES AND CHLAMYDIAE:frequency",
                                           " OTHER INFECTIOUS AND PARASITIC DISEASES:frequency",
                                           " VIRAL DISEASES GENERALLY ACCOMPANIED BY EXANTHEM:frequency",
                                           "CERTAIN TRAUMATIC COMPLICATIONS AND UNSPECIFIED INJURIES:frequency",
                                           "COMPLICATIONS OF SURGICAL AND MEDICAL CARE, NOT ELSEWHERE CLASSIFIED:frequency",
                                           "CONTUSION WITH INTACT SKIN SURFACE:frequency",
                                           "DISLOCATION:frequency",
                                           "FRACTURES:frequency",
                                           "INJURY TO BLOOD VESSELS:frequency",
                                           " INJURY TO NERVES AND SPINAL CORD:frequency",
                                           "INTERNAL INJURY OF THORAX, ABDOMEN, AND PELVIS:frequency",
                                           "INTRACRANIAL INJURY, EXCLUDING THOSE WITH SKULL FRACTURE:frequency",
                                           "LATE EFFECTS OF INJURIES, POISONINGS, TOXIC EFFECTS, AND OTHER EXTERNAL CAUSES:frequency",
                                           "OPEN WOUNDS:frequency",
                                           "OTHER AND UNSPECIFIED EFFECTS OF EXTERNAL CAUSES:frequency",
                                           "POISONING BY DRUGS, MEDICINAL AND BIOLOGICAL SUBSTANCES:frequency",
                                           "SPRAINS AND STRAINS OF JOINTS AND ADJACENT MUSCLES:frequency",
                                           "SUPERFICIAL INJURY:frequency",
                                           "NEUROTIC DISORDERS, PERSONALITY DISORDERS, AND OTHER NONPSYCHOTIC MENTAL DISORDERS:frequency",
                                           "PSYCHOSES:frequency",
                                           "NEOPLASMS:frequency",
                                           " ILL-DEFINED AND UNKNOWN CAUSES OF MORBIDITY AND MORTALITY:frequency",
                                           "NONSPECIFIC ABNORMAL FINDINGS:frequency",
                                           " General symptoms:frequency",
                                           "Other symptoms involving abdomen and pelvis:frequency",
                                           "Symptoms concerning nutrition, metabolism, and development:frequency",
                                           "Symptoms involving cardiovascular system:frequency",
                                           " Symptoms involving digestive system:frequency",
                                           "Symptoms involving head and neck:frequency",
                                           "Symptoms involving nervous and musculoskeletal systems:frequency",
                                           "Symptoms involving respiratory system and other chest symptoms:frequency",
                                           "Symptoms involving skin and other integumentary tissue:frequency",
                                           "Symptoms involving urinary system:frequency",
                                           "COMPLICATIONS MAINLY RELATED TO PREGNANCY:presence",
                                           " COMPLICATIONS OCCURRING MAINLY IN THE COURSE OF LABOR AND DELIVERY:presence",
                                           "NORMAL DELIVERY, AND OTHER INDICATIONS FOR CARE IN PREGNANCY, LABOR, AND DELIVERY:presence",
                                           "Other congenital anomalies of circulatory system:presence",
                                           " Other congenital anomalies of heart:presence",
                                           " Other congenital musculoskeletal anomalies:presence",
                                           "Aplastic anemia and other bone marrow failure syndromes:presence",
                                           " Coagulation defects:presence",
                                           "Diseases of white blood cells:presence",
                                           "Iron deficiency anemias:presence",
                                           "Other and unspecified anemias:presence",
                                           "Other diseases of blood and blood-forming organs:presence",
                                           "Purpura and other hemorrhagic conditions:presence",
                                           "CEREBROVASCULAR DISEASE:presence",
                                           "CHRONIC RHEUMATIC HEART DISEASE:presence",
                                           "DISEASES OF ARTERIES, ARTERIOLES, AND CAPILLARIES:presence",
                                           "DISEASES OF PULMONARY CIRCULATION:presence",
                                           "DISEASES OF VEINS AND LYMPHATICS, AND OTHER DISEASES OF CIRCULATORY SYSTEM:presence",
                                           " HYPERTENSIVE DISEASE:presence",
                                           "ISCHEMIC HEART DISEASE:presence",
                                           " OTHER FORMS OF HEART DISEASE:presence",
                                           "DISEASES OF ESOPHAGUS, STOMACH, AND DUODENUM:presence",
                                           "DISEASES OF ORAL CAVITY, SALIVARY GLANDS, AND JAWS:presence",
                                           "HERNIA OF ABDOMINAL CAVITY:presence",
                                           " NONINFECTIOUS ENTERITIS AND COLITIS:presence",
                                           " OTHER DISEASES OF DIGESTIVE SYSTEM:presence",
                                           "OTHER DISEASES OF INTESTINES AND PERITONEUM:presence",
                                           "DISEASES OF MALE GENITAL ORGANS:presence",
                                           " DISORDERS OF BREAST:presence",
                                           "INFLAMMATORY DISEASE OF FEMALE PELVIC ORGANS:presence",
                                           " NEPHRITIS, NEPHROTIC SYNDROME, AND NEPHROSIS:presence",
                                           " OTHER DISEASES OF URINARY SYSTEM:presence",
                                           "OTHER DISORDERS OF FEMALE GENITAL TRACT:presence",
                                           " ARTHROPATHIES AND RELATED DISORDERS:presence",
                                           "DORSOPATHIES:presence",
                                           "OSTEOPATHIES, CHONDROPATHIES, AND ACQUIRED MUSCULOSKELETAL DEFORMITIES:presence",
                                           "RHEUMATISM, EXCLUDING THE BACK:presence",
                                           "DISEASES OF THE EAR AND MASTOID PROCESS:presence",
                                           "DISORDERS OF THE EYE AND ADNEXA:presence",
                                           "DISORDERS OF THE PERIPHERAL NERVOUS SYSTEM:presence",
                                           "HEREDITARY AND DEGENERATIVE DISEASES OF THE CENTRAL NERVOUS SYSTEM:presence",
                                           " INFLAMMATORY DISEASES OF THE CENTRAL NERVOUS SYSTEM:presence",
                                           "ORGANIC SLEEP DISORDERS:presence",
                                           "OTHER DISORDERS OF THE CENTRAL NERVOUS SYSTEM:presence",
                                           " PAIN:presence",
                                           "ACUTE RESPIRATORY INFECTIONS:presence",
                                           "CHRONIC OBSTRUCTIVE PULMONARY DISEASE AND ALLIED CONDITIONS:presence",
                                           "OTHER DISEASES OF RESPIRATORY SYSTEM:presence",
                                           "OTHER DISEASES OF THE UPPER RESPIRATORY TRACT:presence",
                                           "PNEUMOCONIOSES AND OTHER LUNG DISEASES DUE TO EXTERNAL AGENTS:presence",
                                           " PNEUMONIA AND INFLUENZA:presence",
                                           "INFECTIONS OF SKIN AND SUBCUTANEOUS TISSUE:presence",
                                           "OTHER DISEASES OF SKIN AND SUBCUTANEOUS TISSUE:presence",
                                           "OTHER INFLAMMATORY CONDITIONS OF SKIN AND SUBCUTANEOUS TISSUE:presence",
                                           " DISEASES OF OTHER ENDOCRINE GLANDS:presence",
                                           "DISORDERS OF THYROID GLAND:presence",
                                           " NUTRITIONAL DEFICIENCIES:presence",
                                           "OTHER METABOLIC AND IMMUNITY DISORDERS:presence",
                                           " INTESTINAL INFECTIOUS DISEASES:presence",
                                           "MYCOSES:presence",
                                           " OTHER BACTERIAL DISEASES:presence",
                                           "OTHER DISEASES DUE TO VIRUSES AND CHLAMYDIAE:presence",
                                           " OTHER INFECTIOUS AND PARASITIC DISEASES:presence",
                                           " VIRAL DISEASES GENERALLY ACCOMPANIED BY EXANTHEM:presence",
                                           "CERTAIN TRAUMATIC COMPLICATIONS AND UNSPECIFIED INJURIES:presence",
                                           "COMPLICATIONS OF SURGICAL AND MEDICAL CARE, NOT ELSEWHERE CLASSIFIED:presence",
                                           "CONTUSION WITH INTACT SKIN SURFACE:presence",
                                           "DISLOCATION:presence",
                                           "FRACTURES:presence",
                                           "INTERNAL INJURY OF THORAX, ABDOMEN, AND PELVIS:presence",
                                           "INTRACRANIAL INJURY, EXCLUDING THOSE WITH SKULL FRACTURE:presence",
                                           "LATE EFFECTS OF INJURIES, POISONINGS, TOXIC EFFECTS, AND OTHER EXTERNAL CAUSES:presence",
                                           "OPEN WOUNDS:presence",
                                           "OTHER AND UNSPECIFIED EFFECTS OF EXTERNAL CAUSES:presence",
                                           "POISONING BY DRUGS, MEDICINAL AND BIOLOGICAL SUBSTANCES:presence",
                                           "SPRAINS AND STRAINS OF JOINTS AND ADJACENT MUSCLES:presence",
                                           "SUPERFICIAL INJURY:presence",
                                           "NEUROTIC DISORDERS, PERSONALITY DISORDERS, AND OTHER NONPSYCHOTIC MENTAL DISORDERS:presence",
                                           "PSYCHOSES:presence",
                                           "NEOPLASMS:presence",
                                           " ILL-DEFINED AND UNKNOWN CAUSES OF MORBIDITY AND MORTALITY:presence",
                                           "NONSPECIFIC ABNORMAL FINDINGS:presence",
                                           " General symptoms:presence",
                                           "Other symptoms involving abdomen and pelvis:presence",
                                           "Symptoms concerning nutrition, metabolism, and development:presence",
                                           "Symptoms involving cardiovascular system:presence",
                                           " Symptoms involving digestive system:presence",
                                           "Symptoms involving head and neck:presence",
                                           "Symptoms involving nervous and musculoskeletal systems:presence",
                                           "Symptoms involving respiratory system and other chest symptoms:presence",
                                           "Symptoms involving skin and other integumentary tissue:presence",
                                           "Symptoms involving urinary system:presence",
                                           "albumin:Binary",
                                           "alk:Binary",
                                           "ast:Binary",
                                           "anion:Binary",
                                           "bilirubin:Binary",
                                           "bun:Binary",
                                           "bun_cre:Binary",
                                           "calcium:Binary",
                                           "creatinine:Binary",
                                           "d-dimer:Binary",
                                           "glucose:Binary",
                                           "hemoglobin:Binary",
                                           "a1c:Binary",
                                           "hgb:Binary",
                                           "inr:Binary",
                                           "lactate:Binary",
                                           "platelet:Binary",
                                           "potassium:Binary",
                                           "ptt:Binary",
                                           "sodium:Binary",
                                           "wbc:Binary",
                                           "albumin:Value",
                                           "alk:Value",
                                           "ast:Value",
                                           "anion:Value",
                                           "bilirubin:Value",
                                           "bun:Value",
                                           "bun_cre:Value",
                                           "calcium:Value",
                                           "creatinine:Value",
                                           "d-dimer:Value",
                                           "glucose:Value",
                                           "hemoglobin:Value",
                                           "a1c:Value",
                                           "hgb:Value",
                                           "inr:Value",
                                           "lactate:Value",
                                           "platelet:Value",
                                           "potassium:Value",
                                           "ptt:Value",
                                           "sodium:Value",
                                           "wbc:Value"]]

# Split data into training and validation sets
X_search_pandas: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_search_pandas: pandas.Series = df_train[TARGET_COLUMN]

X_search: numpy.ndarray = numpy.ascontiguousarray(X_search_pandas.to_numpy(), dtype=numpy.float32)
y_search: numpy.ndarray = numpy.ascontiguousarray(y_search_pandas.to_numpy(), dtype=numpy.float32)

feature_names: list[str] = list(X_search_pandas.columns)

cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [5]:
# Load test set
df_test: pandas.DataFrame = pandas.read_csv(CSV_TEST_PATH)
df_test[TARGET_COLUMN] = df_test[TARGET_COLUMN].astype(int)
df_test: pandas.DataFrame = df_test[[      "label",
                                           "Female",
                                           "current_age_yrs",
                                           "SMOKER_Y",
                                           "COMPLICATIONS MAINLY RELATED TO PREGNANCY:frequency",
                                           " COMPLICATIONS OCCURRING MAINLY IN THE COURSE OF LABOR AND DELIVERY:frequency",
                                           "COMPLICATIONS OF THE PUERPERIUM:frequency",
                                           "NORMAL DELIVERY, AND OTHER INDICATIONS FOR CARE IN PREGNANCY, LABOR, AND DELIVERY:frequency",
                                           " Bulbus cordis anomalies and anomalies of cardiac septal closure:frequency",
                                           "Congenital anomalies of urinary system:frequency",
                                           " Other and unspecified congenital anomalies:frequency",
                                           "Other congenital anomalies of circulatory system:frequency",
                                           " Other congenital anomalies of heart:frequency",
                                           "Other congenital anomalies of nervous system:frequency",
                                           " Other congenital musculoskeletal anomalies:frequency",
                                           "Acquired hemolytic anemias:frequency",
                                           "Aplastic anemia and other bone marrow failure syndromes:frequency",
                                           " Coagulation defects:frequency",
                                           "Diseases of white blood cells:frequency",
                                           "Iron deficiency anemias:frequency",
                                           "Other and unspecified anemias:frequency",
                                           "Other diseases of blood and blood-forming organs:frequency",
                                           "Purpura and other hemorrhagic conditions:frequency",
                                           "CEREBROVASCULAR DISEASE:frequency",
                                           "CHRONIC RHEUMATIC HEART DISEASE:frequency",
                                           "DISEASES OF ARTERIES, ARTERIOLES, AND CAPILLARIES:frequency",
                                           "DISEASES OF PULMONARY CIRCULATION:frequency",
                                           "DISEASES OF VEINS AND LYMPHATICS, AND OTHER DISEASES OF CIRCULATORY SYSTEM:frequency",
                                           " HYPERTENSIVE DISEASE:frequency",
                                           "ISCHEMIC HEART DISEASE:frequency",
                                           " OTHER FORMS OF HEART DISEASE:frequency",
                                           " APPENDICITIS:frequency",
                                           "DISEASES OF ESOPHAGUS, STOMACH, AND DUODENUM:frequency",
                                           "DISEASES OF ORAL CAVITY, SALIVARY GLANDS, AND JAWS:frequency",
                                           "HERNIA OF ABDOMINAL CAVITY:frequency",
                                           " NONINFECTIOUS ENTERITIS AND COLITIS:frequency",
                                           " OTHER DISEASES OF DIGESTIVE SYSTEM:frequency",
                                           "OTHER DISEASES OF INTESTINES AND PERITONEUM:frequency",
                                           "DISEASES OF MALE GENITAL ORGANS:frequency",
                                           " DISORDERS OF BREAST:frequency",
                                           "INFLAMMATORY DISEASE OF FEMALE PELVIC ORGANS:frequency",
                                           " NEPHRITIS, NEPHROTIC SYNDROME, AND NEPHROSIS:frequency",
                                           " OTHER DISEASES OF URINARY SYSTEM:frequency",
                                           "OTHER DISORDERS OF FEMALE GENITAL TRACT:frequency",
                                           " ARTHROPATHIES AND RELATED DISORDERS:frequency",
                                           "DORSOPATHIES:frequency",
                                           "OSTEOPATHIES, CHONDROPATHIES, AND ACQUIRED MUSCULOSKELETAL DEFORMITIES:frequency",
                                           "RHEUMATISM, EXCLUDING THE BACK:frequency",
                                           "DISEASES OF THE EAR AND MASTOID PROCESS:frequency",
                                           "DISORDERS OF THE EYE AND ADNEXA:frequency",
                                           "DISORDERS OF THE PERIPHERAL NERVOUS SYSTEM:frequency",
                                           "HEREDITARY AND DEGENERATIVE DISEASES OF THE CENTRAL NERVOUS SYSTEM:frequency",
                                           " INFLAMMATORY DISEASES OF THE CENTRAL NERVOUS SYSTEM:frequency",
                                           "ORGANIC SLEEP DISORDERS:frequency",
                                           "OTHER DISORDERS OF THE CENTRAL NERVOUS SYSTEM:frequency",
                                           " PAIN:frequency",
                                           "ACUTE RESPIRATORY INFECTIONS:frequency",
                                           "CHRONIC OBSTRUCTIVE PULMONARY DISEASE AND ALLIED CONDITIONS:frequency",
                                           "OTHER DISEASES OF RESPIRATORY SYSTEM:frequency",
                                           "OTHER DISEASES OF THE UPPER RESPIRATORY TRACT:frequency",
                                           "PNEUMOCONIOSES AND OTHER LUNG DISEASES DUE TO EXTERNAL AGENTS:frequency",
                                           " PNEUMONIA AND INFLUENZA:frequency",
                                           "INFECTIONS OF SKIN AND SUBCUTANEOUS TISSUE:frequency",
                                           "OTHER DISEASES OF SKIN AND SUBCUTANEOUS TISSUE:frequency",
                                           "OTHER INFLAMMATORY CONDITIONS OF SKIN AND SUBCUTANEOUS TISSUE:frequency",
                                           " DISEASES OF OTHER ENDOCRINE GLANDS:frequency",
                                           "DISORDERS OF THYROID GLAND:frequency",
                                           " NUTRITIONAL DEFICIENCIES:frequency",
                                           "OTHER METABOLIC AND IMMUNITY DISORDERS:frequency",
                                           " INTESTINAL INFECTIOUS DISEASES:frequency",
                                           "MYCOSES:frequency",
                                           " OTHER BACTERIAL DISEASES:frequency",
                                           "OTHER DISEASES DUE TO VIRUSES AND CHLAMYDIAE:frequency",
                                           " OTHER INFECTIOUS AND PARASITIC DISEASES:frequency",
                                           " VIRAL DISEASES GENERALLY ACCOMPANIED BY EXANTHEM:frequency",
                                           "CERTAIN TRAUMATIC COMPLICATIONS AND UNSPECIFIED INJURIES:frequency",
                                           "COMPLICATIONS OF SURGICAL AND MEDICAL CARE, NOT ELSEWHERE CLASSIFIED:frequency",
                                           "CONTUSION WITH INTACT SKIN SURFACE:frequency",
                                           "DISLOCATION:frequency",
                                           "FRACTURES:frequency",
                                           "INJURY TO BLOOD VESSELS:frequency",
                                           " INJURY TO NERVES AND SPINAL CORD:frequency",
                                           "INTERNAL INJURY OF THORAX, ABDOMEN, AND PELVIS:frequency",
                                           "INTRACRANIAL INJURY, EXCLUDING THOSE WITH SKULL FRACTURE:frequency",
                                           "LATE EFFECTS OF INJURIES, POISONINGS, TOXIC EFFECTS, AND OTHER EXTERNAL CAUSES:frequency",
                                           "OPEN WOUNDS:frequency",
                                           "OTHER AND UNSPECIFIED EFFECTS OF EXTERNAL CAUSES:frequency",
                                           "POISONING BY DRUGS, MEDICINAL AND BIOLOGICAL SUBSTANCES:frequency",
                                           "SPRAINS AND STRAINS OF JOINTS AND ADJACENT MUSCLES:frequency",
                                           "SUPERFICIAL INJURY:frequency",
                                           "NEUROTIC DISORDERS, PERSONALITY DISORDERS, AND OTHER NONPSYCHOTIC MENTAL DISORDERS:frequency",
                                           "PSYCHOSES:frequency",
                                           "NEOPLASMS:frequency",
                                           " ILL-DEFINED AND UNKNOWN CAUSES OF MORBIDITY AND MORTALITY:frequency",
                                           "NONSPECIFIC ABNORMAL FINDINGS:frequency",
                                           " General symptoms:frequency",
                                           "Other symptoms involving abdomen and pelvis:frequency",
                                           "Symptoms concerning nutrition, metabolism, and development:frequency",
                                           "Symptoms involving cardiovascular system:frequency",
                                           " Symptoms involving digestive system:frequency",
                                           "Symptoms involving head and neck:frequency",
                                           "Symptoms involving nervous and musculoskeletal systems:frequency",
                                           "Symptoms involving respiratory system and other chest symptoms:frequency",
                                           "Symptoms involving skin and other integumentary tissue:frequency",
                                           "Symptoms involving urinary system:frequency",
                                           "COMPLICATIONS MAINLY RELATED TO PREGNANCY:presence",
                                           " COMPLICATIONS OCCURRING MAINLY IN THE COURSE OF LABOR AND DELIVERY:presence",
                                           "NORMAL DELIVERY, AND OTHER INDICATIONS FOR CARE IN PREGNANCY, LABOR, AND DELIVERY:presence",
                                           "Other congenital anomalies of circulatory system:presence",
                                           " Other congenital anomalies of heart:presence",
                                           " Other congenital musculoskeletal anomalies:presence",
                                           "Aplastic anemia and other bone marrow failure syndromes:presence",
                                           " Coagulation defects:presence",
                                           "Diseases of white blood cells:presence",
                                           "Iron deficiency anemias:presence",
                                           "Other and unspecified anemias:presence",
                                           "Other diseases of blood and blood-forming organs:presence",
                                           "Purpura and other hemorrhagic conditions:presence",
                                           "CEREBROVASCULAR DISEASE:presence",
                                           "CHRONIC RHEUMATIC HEART DISEASE:presence",
                                           "DISEASES OF ARTERIES, ARTERIOLES, AND CAPILLARIES:presence",
                                           "DISEASES OF PULMONARY CIRCULATION:presence",
                                           "DISEASES OF VEINS AND LYMPHATICS, AND OTHER DISEASES OF CIRCULATORY SYSTEM:presence",
                                           " HYPERTENSIVE DISEASE:presence",
                                           "ISCHEMIC HEART DISEASE:presence",
                                           " OTHER FORMS OF HEART DISEASE:presence",
                                           "DISEASES OF ESOPHAGUS, STOMACH, AND DUODENUM:presence",
                                           "DISEASES OF ORAL CAVITY, SALIVARY GLANDS, AND JAWS:presence",
                                           "HERNIA OF ABDOMINAL CAVITY:presence",
                                           " NONINFECTIOUS ENTERITIS AND COLITIS:presence",
                                           " OTHER DISEASES OF DIGESTIVE SYSTEM:presence",
                                           "OTHER DISEASES OF INTESTINES AND PERITONEUM:presence",
                                           "DISEASES OF MALE GENITAL ORGANS:presence",
                                           " DISORDERS OF BREAST:presence",
                                           "INFLAMMATORY DISEASE OF FEMALE PELVIC ORGANS:presence",
                                           " NEPHRITIS, NEPHROTIC SYNDROME, AND NEPHROSIS:presence",
                                           " OTHER DISEASES OF URINARY SYSTEM:presence",
                                           "OTHER DISORDERS OF FEMALE GENITAL TRACT:presence",
                                           " ARTHROPATHIES AND RELATED DISORDERS:presence",
                                           "DORSOPATHIES:presence",
                                           "OSTEOPATHIES, CHONDROPATHIES, AND ACQUIRED MUSCULOSKELETAL DEFORMITIES:presence",
                                           "RHEUMATISM, EXCLUDING THE BACK:presence",
                                           "DISEASES OF THE EAR AND MASTOID PROCESS:presence",
                                           "DISORDERS OF THE EYE AND ADNEXA:presence",
                                           "DISORDERS OF THE PERIPHERAL NERVOUS SYSTEM:presence",
                                           "HEREDITARY AND DEGENERATIVE DISEASES OF THE CENTRAL NERVOUS SYSTEM:presence",
                                           " INFLAMMATORY DISEASES OF THE CENTRAL NERVOUS SYSTEM:presence",
                                           "ORGANIC SLEEP DISORDERS:presence",
                                           "OTHER DISORDERS OF THE CENTRAL NERVOUS SYSTEM:presence",
                                           " PAIN:presence",
                                           "ACUTE RESPIRATORY INFECTIONS:presence",
                                           "CHRONIC OBSTRUCTIVE PULMONARY DISEASE AND ALLIED CONDITIONS:presence",
                                           "OTHER DISEASES OF RESPIRATORY SYSTEM:presence",
                                           "OTHER DISEASES OF THE UPPER RESPIRATORY TRACT:presence",
                                           "PNEUMOCONIOSES AND OTHER LUNG DISEASES DUE TO EXTERNAL AGENTS:presence",
                                           " PNEUMONIA AND INFLUENZA:presence",
                                           "INFECTIONS OF SKIN AND SUBCUTANEOUS TISSUE:presence",
                                           "OTHER DISEASES OF SKIN AND SUBCUTANEOUS TISSUE:presence",
                                           "OTHER INFLAMMATORY CONDITIONS OF SKIN AND SUBCUTANEOUS TISSUE:presence",
                                           " DISEASES OF OTHER ENDOCRINE GLANDS:presence",
                                           "DISORDERS OF THYROID GLAND:presence",
                                           " NUTRITIONAL DEFICIENCIES:presence",
                                           "OTHER METABOLIC AND IMMUNITY DISORDERS:presence",
                                           " INTESTINAL INFECTIOUS DISEASES:presence",
                                           "MYCOSES:presence",
                                           " OTHER BACTERIAL DISEASES:presence",
                                           "OTHER DISEASES DUE TO VIRUSES AND CHLAMYDIAE:presence",
                                           " OTHER INFECTIOUS AND PARASITIC DISEASES:presence",
                                           " VIRAL DISEASES GENERALLY ACCOMPANIED BY EXANTHEM:presence",
                                           "CERTAIN TRAUMATIC COMPLICATIONS AND UNSPECIFIED INJURIES:presence",
                                           "COMPLICATIONS OF SURGICAL AND MEDICAL CARE, NOT ELSEWHERE CLASSIFIED:presence",
                                           "CONTUSION WITH INTACT SKIN SURFACE:presence",
                                           "DISLOCATION:presence",
                                           "FRACTURES:presence",
                                           "INTERNAL INJURY OF THORAX, ABDOMEN, AND PELVIS:presence",
                                           "INTRACRANIAL INJURY, EXCLUDING THOSE WITH SKULL FRACTURE:presence",
                                           "LATE EFFECTS OF INJURIES, POISONINGS, TOXIC EFFECTS, AND OTHER EXTERNAL CAUSES:presence",
                                           "OPEN WOUNDS:presence",
                                           "OTHER AND UNSPECIFIED EFFECTS OF EXTERNAL CAUSES:presence",
                                           "POISONING BY DRUGS, MEDICINAL AND BIOLOGICAL SUBSTANCES:presence",
                                           "SPRAINS AND STRAINS OF JOINTS AND ADJACENT MUSCLES:presence",
                                           "SUPERFICIAL INJURY:presence",
                                           "NEUROTIC DISORDERS, PERSONALITY DISORDERS, AND OTHER NONPSYCHOTIC MENTAL DISORDERS:presence",
                                           "PSYCHOSES:presence",
                                           "NEOPLASMS:presence",
                                           " ILL-DEFINED AND UNKNOWN CAUSES OF MORBIDITY AND MORTALITY:presence",
                                           "NONSPECIFIC ABNORMAL FINDINGS:presence",
                                           " General symptoms:presence",
                                           "Other symptoms involving abdomen and pelvis:presence",
                                           "Symptoms concerning nutrition, metabolism, and development:presence",
                                           "Symptoms involving cardiovascular system:presence",
                                           " Symptoms involving digestive system:presence",
                                           "Symptoms involving head and neck:presence",
                                           "Symptoms involving nervous and musculoskeletal systems:presence",
                                           "Symptoms involving respiratory system and other chest symptoms:presence",
                                           "Symptoms involving skin and other integumentary tissue:presence",
                                           "Symptoms involving urinary system:presence",
                                           "albumin:Binary",
                                           "alk:Binary",
                                           "ast:Binary",
                                           "anion:Binary",
                                           "bilirubin:Binary",
                                           "bun:Binary",
                                           "bun_cre:Binary",
                                           "calcium:Binary",
                                           "creatinine:Binary",
                                           "d-dimer:Binary",
                                           "glucose:Binary",
                                           "hemoglobin:Binary",
                                           "a1c:Binary",
                                           "hgb:Binary",
                                           "inr:Binary",
                                           "lactate:Binary",
                                           "platelet:Binary",
                                           "potassium:Binary",
                                           "ptt:Binary",
                                           "sodium:Binary",
                                           "wbc:Binary",
                                           "albumin:Value",
                                           "alk:Value",
                                           "ast:Value",
                                           "anion:Value",
                                           "bilirubin:Value",
                                           "bun:Value",
                                           "bun_cre:Value",
                                           "calcium:Value",
                                           "creatinine:Value",
                                           "d-dimer:Value",
                                           "glucose:Value",
                                           "hemoglobin:Value",
                                           "a1c:Value",
                                           "hgb:Value",
                                           "inr:Value",
                                           "lactate:Value",
                                           "platelet:Value",
                                           "potassium:Value",
                                           "ptt:Value",
                                           "sodium:Value",
                                           "wbc:Value"]]

y_test: pandas.Series = df_test[TARGET_COLUMN]
X_test: pandas.DataFrame = df_test.drop(columns=[TARGET_COLUMN])

In [6]:
# Pre-compute folds and scaling
fold_indices: list[tuple[numpy.ndarray, numpy.ndarray]] = list(cv.split(X_search, y_search))

X_train_scaled_folds: list[numpy.ndarray] = []
X_val_scaled_folds: list[numpy.ndarray] = []
y_train_folds: list[numpy.ndarray] = []
y_val_folds: list[numpy.ndarray] = []

n_splits: int = len(fold_indices)
n_features: int = len(feature_names)
corr_matrix: numpy.ndarray = numpy.zeros((n_splits, n_features), dtype=float)
for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
    X_fold_train: numpy.ndarray = X_search[train_idx]
    X_fold_val: numpy.ndarray = X_search[val_idx]
    y_fold_train: numpy.ndarray = y_search[train_idx]
    y_fold_val: numpy.ndarray = y_search[val_idx]

    # Pre-scale (all features at once)
    scaler: StandardScaler = StandardScaler()
    X_train_scaled_folds.append(scaler.fit_transform(X_fold_train))
    X_val_scaled_folds.append(scaler.transform(X_fold_val))
    y_train_folds.append(y_fold_train)
    y_val_folds.append(y_fold_val)

    # Calculate Point-biserial correlation coefficients (target value is binary and the input variables are continuous)
    # Calculate the average correlation for each feature across all fold
    for col_idx in range(X_fold_train.shape[1]):
        feature_col: numpy.ndarray = X_fold_train[:, col_idx]
        unique_vals: int = len(numpy.unique(feature_col))

        if unique_vals <= 1:
            corr: float = 0.0
        elif unique_vals == 2:
            corr: float = matthews_corrcoef(y_fold_train, feature_col.astype(int))
        else:
            corr, _ = pointbiserialr(y_fold_train, feature_col)

        corr_matrix[fold_idx, col_idx] = corr

In [7]:
training_results_multi: dict[int, list[list[creator.Individual]]] = {}
training_results_single: dict[int, list[creator.IndividualSingle]] = {}

seeds: list[int] = [42]#, 123, 5678, 7286816]
for s in seeds:
    set_seed(s)
    multi_objective_training: MultiObjectiveTraining = MultiObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_MULTI),
        feature_names=feature_names,
        fold_indices=fold_indices,
        X_train_scaled_folds=X_train_scaled_folds,
        X_val_scaled_folds=X_val_scaled_folds,
        y_train_folds=y_train_folds,
        y_val_folds=y_val_folds,
        corr_matrix=corr_matrix,
    )
    pareto_front: list[creator.Individual] = multi_objective_training.run()
    training_results_multi[s] = pareto_front
    multi_objective_training.clear_cache()

    set_seed(s)
    single_objective_training: SingleObjectiveTraining = SingleObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_SINGLE),
        feature_names=feature_names,
        fold_indices=fold_indices,
        X_train_scaled_folds=X_train_scaled_folds,
        X_val_scaled_folds=X_val_scaled_folds,
        y_train_folds=y_train_folds,
        y_val_folds=y_val_folds)
    best_indi: creator.IndividualSingle = single_objective_training.run()
    training_results_single[s] = best_indi
    single_objective_training.clear_cache()

Generation 1 done | Pareto size: 10
Generation 2 done | Pareto size: 11
Generation 3 done | Pareto size: 13
Generation 4 done | Pareto size: 16
Generation 5 done | Pareto size: 14
Generation 6 done | Pareto size: 13
Generation 7 done | Pareto size: 11
Generation 8 done | Pareto size: 15
Generation 9 done | Pareto size: 18
Generation 10 done | Pareto size: 10
Generation 11 done | Pareto size: 11
Generation 12 done | Pareto size: 16
Generation 13 done | Pareto size: 19
Generation 14 done | Pareto size: 14
Generation 15 done | Pareto size: 14
Generation 16 done | Pareto size: 11
Generation 17 done | Pareto size: 12
Generation 18 done | Pareto size: 13
Generation 19 done | Pareto size: 15
Generation 20 done | Pareto size: 17
Generation 21 done | Pareto size: 13
Generation 22 done | Pareto size: 12
Generation 23 done | Pareto size: 12
Generation 24 done | Pareto size: 14
Generation 25 done | Pareto size: 18
Generation 26 done | Pareto size: 17
Generation 27 done | Pareto size: 20
Generation

## Evaluation: Single vs Multi-Objective Robustness Comparison

For each seed we:
1. Retrain a final LogisticRegression on the **full training set** using features selected by the GA.
   - Multi-objective: the Pareto individual with the **highest sign-consistency** score.
   - Single-objective: the best individual from the Hall of Fame.
2. Evaluate on clean data and under increasing noise levels on **both test and validation (training) sets**:
   - **Gaussian noise + mean shift** on continuous features
   - **Random corruption noise** on dummy (binary) features
3. Save CSV results and comparison plots to the result directory (separate files per dataset).

In [8]:
# Detect column types automatically from the training DataFrame
X_train_df: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_train_series: pandas.Series = df_train[TARGET_COLUMN]

continuous_cols: list[str] = get_continuous_columns(X_train_df)
dummy_cols: list[str] = get_dummy_columns(X_train_df)

# Training-set std for proportional noise scaling
train_std: pandas.Series = X_train_df.std()

print(f"Continuous features: {len(continuous_cols)}")
print(f"Dummy features:     {len(dummy_cols)}")

Continuous features: 123
Dummy features:     115


In [9]:
# Multi-objective: pick the Pareto individual with the highest sign-consistency
evaluation_results_test: dict[int, pandas.DataFrame] = {}

for s in seeds:
    set_seed(s)

    # --- Multi-objective: select max sign-consistency or knee point individual from Pareto front ---
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = pareto_front[knee_point_index(pareto_front) if USE_KNEE_POINT_SELECTION else best_sign_consistency_index(pareto_front)]
    print(f"Seed {s} | Selected multi-objective individual: "
          f"AUC={best_multi_ind.fitness.values[0]:.4f}, "
          f"SignConsistency={best_multi_ind.fitness.values[1]:.4f}, "
          f"Features={sum(best_multi_ind)}")

    # --- Build final model packages on full training set ---
    model_pkg_multi = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    model_pkg_single = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    # --- Run noise evaluation on TEST set and save plots/CSV ---
    eval_df = evaluate_and_save(
        seed=s,
        model_pkg_multi=model_pkg_multi,
        model_pkg_single=model_pkg_single,
        X_eval=X_test,
        y_eval=y_test,
        train_std=train_std,
        continuous_cols=continuous_cols,
        dummy_cols=dummy_cols,
        result_directory=RESULT_PATH_EVAL,
        use_roc_auc=True,
        dataset_label="Test",
    )
    evaluation_results_test[s] = eval_df

print("\nTest set evaluation complete. Results saved to:", RESULT_PATH_EVAL)

Seed 42 | Selected multi-objective individual: AUC=0.9385, SignConsistency=0.9444, Features=66
  Seed 42 [Test] | Multi features:  66 | Single features:  67 | Clean AUC  Multi: 0.9047  Single: 0.8936

Test set evaluation complete. Results saved to: 2026-04-21_09-28-39\evaluation


In [10]:
# Noise evaluation on VALIDATION (training) set
evaluation_results_val: dict[int, pandas.DataFrame] = {}

for s in seeds:
    set_seed(s)

    # --- Multi-objective: select max sign-consistency or knee point individual from Pareto front ---
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = pareto_front[knee_point_index(pareto_front) if USE_KNEE_POINT_SELECTION else best_sign_consistency_index(pareto_front)]

    # --- Build final model packages on full training set ---
    model_pkg_multi = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    model_pkg_single = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    # --- Run noise evaluation on VALIDATION (training) set and save plots/CSV ---
    eval_df = evaluate_and_save(
        seed=s,
        model_pkg_multi=model_pkg_multi,
        model_pkg_single=model_pkg_single,
        X_eval=X_train_df,
        y_eval=y_train_series,
        train_std=train_std,
        continuous_cols=continuous_cols,
        dummy_cols=dummy_cols,
        result_directory=RESULT_PATH_EVAL,
        use_roc_auc=True,
        dataset_label="Validation",
    )
    evaluation_results_val[s] = eval_df

print("\nValidation set evaluation complete. Results saved to:", RESULT_PATH_EVAL)

  Seed 42 [Validation] | Multi features:  66 | Single features:  67 | Clean AUC  Multi: 0.9519  Single: 0.9537

Validation set evaluation complete. Results saved to: 2026-04-21_09-28-39\evaluation


## Feature Selection Stability (Jaccard Similarity)

Compare how consistently each method selects the same features across different random seeds.
- **Jaccard similarity** = |intersection| / |union| between two feature sets (1.0 = identical, 0.0 = no overlap).
- A **pairwise heatmap** shows seed-to-seed stability for multi-objective vs single-objective.
- A **feature frequency bar chart** shows which features are consistently selected across seeds.

In [11]:
# Collect the best individual per seed for both methods
best_multi_by_seed: dict[int, list[int]] = {}
best_single_by_seed: dict[int, list[int]] = {}

for s in seeds:
    # Multi-objective: max sign-consistency or knee point (same selection as evaluation)
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = pareto_front[knee_point_index(pareto_front) if USE_KNEE_POINT_SELECTION else best_sign_consistency_index(pareto_front)]
    best_multi_by_seed[s]: list[int] = list(best_multi_ind)

    # Single-objective: best individual from HoF
    best_single_by_seed[s]: list[int] = list(training_results_single[s])

# Compute pairwise Jaccard stability matrices
seeds_multi, matrix_multi = compute_stability_matrix(best_multi_by_seed, feature_names)
seeds_single, matrix_single = compute_stability_matrix(best_single_by_seed, feature_names)

# Print summary
print("=== Feature Selection Stability (Jaccard Similarity) ===\n")
print("Multi-Objective (SiCo-MOGA) pairwise Jaccard:")
for i, si in enumerate(seeds_multi):
    for j, sj in enumerate(seeds_multi):
        if j > i:
            print(f"  Seed {si} vs Seed {sj}: {matrix_multi[i, j]:.3f}")

print("\nSingle-Objective (AUC-only GA) pairwise Jaccard:")
for i, si in enumerate(seeds_single):
    for j, sj in enumerate(seeds_single):
        if j > i:
            print(f"  Seed {si} vs Seed {sj}: {matrix_single[i, j]:.3f}")

# Save plots
stability_dir: str = os.path.join(RESULT_PATH_EVAL, "stability")

plot_stability_heatmap(
    seeds_multi, matrix_multi,
    seeds_single, matrix_single,
    os.path.join(stability_dir, "jaccard_heatmap.png"))

plot_feature_frequency(
    best_multi_by_seed, feature_names,
    "Multi-Objective (SiCo-MOGA)",
    os.path.join(stability_dir, "feature_frequency_multi.png"))

plot_feature_frequency(
    best_single_by_seed, feature_names,
    "Single-Objective (AUC-only GA)",
    os.path.join(stability_dir, "feature_frequency_single.png"))

print(f"\nStability plots saved to: {stability_dir}")

=== Feature Selection Stability (Jaccard Similarity) ===

Multi-Objective (SiCo-MOGA) pairwise Jaccard:

Single-Objective (AUC-only GA) pairwise Jaccard:

Stability plots saved to: 2026-04-21_09-28-39\evaluation\stability


## Multicollinearity Analysis: Variance Inflation Factor (VIF)

VIF measures how much the variance of a regression coefficient is inflated due to multicollinearity.
- **VIF = 1**: no correlation with other predictors
- **VIF = 5**: moderate multicollinearity
- **VIF > 10**: high multicollinearity (coefficient estimates become unreliable)

We compare the VIF distributions of features selected by SOGA (Single-Objective) vs MOGA (Multi-Objective) across all seeds.

In [12]:
# Compute VIF for features selected by each method across seeds
vif_multi: pandas.DataFrame = compute_vif_across_seeds(best_multi_by_seed, feature_names, X_train_df)
vif_single: pandas.DataFrame = compute_vif_across_seeds(best_single_by_seed, feature_names, X_train_df)

print("=== Variance Inflation Factor (VIF) Summary ===\n")
print("Multi-Objective (SiCo-MOGA):")
print(f"  Features per seed: {vif_multi.groupby('seed')['feature'].count().values}")
print(f"  Median VIF: {vif_multi['vif'].median():.2f}")
print(f"  Mean VIF:   {vif_multi['vif'].mean():.2f}")
print(f"  Max VIF:    {vif_multi['vif'].max():.2f}")
print(f"  Features with VIF > 5: {(vif_multi['vif'] > 5).sum()} / {len(vif_multi)}")

print("\nSingle-Objective (AUC-only GA):")
print(f"  Features per seed: {vif_single.groupby('seed')['feature'].count().values}")
print(f"  Median VIF: {vif_single['vif'].median():.2f}")
print(f"  Mean VIF:   {vif_single['vif'].mean():.2f}")
print(f"  Max VIF:    {vif_single['vif'].max():.2f}")
print(f"  Features with VIF > 5: {(vif_single['vif'] > 5).sum()} / {len(vif_single)}")

# Save VIF plot
vif_dir: str = os.path.join(RESULT_PATH_EVAL, "vif")
plot_vif_boxplot(vif_multi, vif_single, os.path.join(vif_dir, "vif_comparison.png"))

# Save VIF data as CSV
vif_multi.to_csv(os.path.join(vif_dir, "vif_multi.csv"), index=False)
vif_single.to_csv(os.path.join(vif_dir, "vif_single.csv"), index=False)

print(f"\nVIF plots and data saved to: {vif_dir}")

=== Variance Inflation Factor (VIF) Summary ===

Multi-Objective (SiCo-MOGA):
  Features per seed: [66]
  Median VIF: 1.30
  Mean VIF:   1.66
  Max VIF:    7.63
  Features with VIF > 5: 3 / 66

Single-Objective (AUC-only GA):
  Features per seed: [67]
  Median VIF: 1.28
  Mean VIF:   2.45
  Max VIF:    28.01
  Features with VIF > 5: 3 / 67

VIF plots and data saved to: 2026-04-21_09-28-39\evaluation\vif


## Feature Set Comparison: Multi-Objective vs Single-Objective

For each seed, compare the features selected by SiCo-MOGA vs. SOGA:
- **Only in Multi**: features that the multi-objective GA selected but single-objective did not
- **Only in Single**: features that the single-objective GA selected but multi-objective did not
- **Common**: features selected by both

This highlights which features the sign-consistency penalty specifically filters out (only-in-single) or adds back (only-in-multi). The results are printed per seed and saved as a CSV.

In [13]:
# Feature set comparison: multi vs single objective
from evaluation_utils import ensure_directory as _ensure_directory

comparison_rows: list[dict] = []

print("=== Feature Set Comparison: Multi-Objective vs Single-Objective ===\n")
for s in seeds:
    multi_bits: list[int] = best_multi_by_seed[s]
    single_bits: list[int] = best_single_by_seed[s]

    multi_set: set[str] = {f for f, b in zip(feature_names, multi_bits) if b == 1}
    single_set: set[str] = {f for f, b in zip(feature_names, single_bits) if b == 1}

    only_in_multi: list[str] = sorted(multi_set - single_set)
    only_in_single: list[str] = sorted(single_set - multi_set)
    common: list[str] = sorted(multi_set & single_set)

    print(f"--- Seed {s} ---")
    print(f"  Multi features: {len(multi_set)} | Single features: {len(single_set)} | Common: {len(common)}")
    print(f"  Only in Multi  ({len(only_in_multi)}):")
    for f in only_in_multi:
        print(f"    + {f}")
    print(f"  Only in Single ({len(only_in_single)}):")
    for f in only_in_single:
        print(f"    - {f}")
    print()

    for f in only_in_multi:
        comparison_rows.append({"seed": s, "feature": f, "membership": "only_in_multi"})
    for f in only_in_single:
        comparison_rows.append({"seed": s, "feature": f, "membership": "only_in_single"})
    for f in common:
        comparison_rows.append({"seed": s, "feature": f, "membership": "common"})

comparison_df: pandas.DataFrame = pandas.DataFrame(comparison_rows)

comparison_dir: str = os.path.join(RESULT_PATH_EVAL, "feature_comparison")
_ensure_directory(comparison_dir)
comparison_df.to_csv(os.path.join(comparison_dir, "feature_set_comparison.csv"), index=False)

print(f"Feature set comparison CSV saved to: {comparison_dir}")

=== Feature Set Comparison: Multi-Objective vs Single-Objective ===

--- Seed 42 ---
  Multi features: 66 | Single features: 67 | Common: 33
  Only in Multi  (33):
    +  Bulbus cordis anomalies and anomalies of cardiac septal closure:frequency
    +  COMPLICATIONS OCCURRING MAINLY IN THE COURSE OF LABOR AND DELIVERY:frequency
    +  DISEASES OF OTHER ENDOCRINE GLANDS:frequency
    +  HYPERTENSIVE DISEASE:frequency
    +  NEPHRITIS, NEPHROTIC SYNDROME, AND NEPHROSIS:frequency
    +  OTHER FORMS OF HEART DISEASE:frequency
    +  PAIN:frequency
    +  PNEUMONIA AND INFLUENZA:frequency
    +  VIRAL DISEASES GENERALLY ACCOMPANIED BY EXANTHEM:presence
    + CEREBROVASCULAR DISEASE:presence
    + DISEASES OF ARTERIES, ARTERIOLES, AND CAPILLARIES:frequency
    + DISEASES OF THE EAR AND MASTOID PROCESS:presence
    + DORSOPATHIES:frequency
    + FRACTURES:presence
    + INFECTIONS OF SKIN AND SUBCUTANEOUS TISSUE:frequency
    + INFECTIONS OF SKIN AND SUBCUTANEOUS TISSUE:presence
    + INJURY T

## Sign Inconsistency in Single-Objective Model

For each feature selected by the Single-Objective GA, compare:
- **Marginal correlation sign** with the target (point-biserial for continuous, Matthews for binary), computed on the full training set
- **Logistic regression coefficient sign**, from the final model retrained on the full training set

Features where these signs **disagree** are sign-inconsistent: the feature's individual association with the target is opposite to how the model uses it (a hallmark of suppressor effects and multicollinearity). The sign-consistency objective in SiCo-MOGA is designed to avoid exactly these cases.

In [14]:
# Sign inconsistency in Single-Objective model: correlation sign vs coefficient sign

# Pre-compute marginal correlations with target on the FULL training set
# Cast to int for matthews_corrcoef so sklearn's type_of_target detects it as
# binary rather than continuous (ValueError otherwise when the column dtype is float).
y_full: numpy.ndarray = y_train_series.to_numpy().astype(int)

full_train_corr: dict[str, float] = {}
for col in feature_names:
    feature_col: numpy.ndarray = X_train_df[col].to_numpy()
    unique_vals: int = len(numpy.unique(feature_col))
    if unique_vals <= 1:
        full_train_corr[col] = 0.0
    elif unique_vals == 2:
        full_train_corr[col] = float(matthews_corrcoef(y_full, feature_col.astype(int)))
    else:
        c, _ = pointbiserialr(y_full, feature_col)
        full_train_corr[col] = float(c)


def _sign(x: float, atol: float = 1e-12) -> int:
    """Return -1/0/+1 with a tolerance around zero."""
    if numpy.isclose(x, 0.0, atol=atol):
        return 0
    return 1 if x > 0 else -1


inconsistency_rows: list[dict] = []

print("=== Sign Inconsistency in Single-Objective Selected Features ===\n")
for s in seeds:
    set_seed(s)
    model_pkg = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    selected: list[str] = model_pkg["features"]
    coefs: numpy.ndarray = model_pkg["model"].coef_[0]

    inconsistent: list[dict] = []
    for feat, coef in zip(selected, coefs):
        corr: float = full_train_corr[feat]
        corr_sign: int = _sign(corr)
        coef_sign: int = _sign(float(coef))
        mismatch: bool = (corr_sign != 0 and coef_sign != 0 and corr_sign != coef_sign)
        if mismatch:
            inconsistent.append({
                "seed": s,
                "feature": feat,
                "correlation": corr,
                "coefficient": float(coef),
                "corr_sign": corr_sign,
                "coef_sign": coef_sign,
            })

    print(f"--- Seed {s} ---")
    print(f"  Single-objective selected features: {len(selected)}")
    print(f"  Sign-inconsistent features:         {len(inconsistent)} "
          f"({100.0 * len(inconsistent) / max(len(selected), 1):.1f}%)")
    if inconsistent:
        print(f"  {'Feature':<40} {'Correlation':>12} {'Coefficient':>14}")
        for row in sorted(inconsistent, key=lambda r: abs(r["coefficient"]), reverse=True):
            print(f"    {row['feature']:<38} {row['correlation']:>12.4f} {row['coefficient']:>14.4f}")
    print()

    inconsistency_rows.extend(inconsistent)

inconsistency_df: pandas.DataFrame = pandas.DataFrame(inconsistency_rows)

inconsistency_dir: str = os.path.join(RESULT_PATH_EVAL, "feature_comparison")
_ensure_directory(inconsistency_dir)
inconsistency_df.to_csv(os.path.join(inconsistency_dir, "single_sign_inconsistency.csv"), index=False)

print(f"Sign inconsistency CSV saved to: {inconsistency_dir}")

=== Sign Inconsistency in Single-Objective Selected Features ===

--- Seed 42 ---
  Single-objective selected features: 67
  Sign-inconsistent features:         16 (23.9%)
  Feature                                   Correlation    Coefficient
    ast:Binary                                   0.0888        -0.4551
    creatinine:Binary                            0.1051        -0.4415
    NEOPLASMS:frequency                         -0.0076         0.2643
    PSYCHOSES:presence                           0.0251        -0.1897
     OTHER BACTERIAL DISEASES:presence           0.0031        -0.1646
    OTHER INFLAMMATORY CONDITIONS OF SKIN AND SUBCUTANEOUS TISSUE:presence      -0.0034         0.1437
    RHEUMATISM, EXCLUDING THE BACK:presence       0.0052        -0.1429
    DISORDERS OF THE PERIPHERAL NERVOUS SYSTEM:frequency       0.0193        -0.1380
    Symptoms involving head and neck:presence      -0.0297         0.1206
     OTHER DISEASES OF URINARY SYSTEM:presence       0.0167        -

## Baseline: Logistic Regression with ALL features (no feature selection)

Train a logistic regression using **every** available feature (no GA, no stepwise search) and report clean AUC/AP on the test set per seed. Serves as the trivial "no selection" reference point against which both the GA-based methods and the forward-stepwise baseline must justify themselves.

In [ ]:
# Baseline 1: Logistic Regression using ALL features (no feature selection)
from sklearn.metrics import roc_auc_score, average_precision_score

baseline_all_rows: list[dict] = []
all_features_ind: list[int] = [1] * len(feature_names)

print("=== Baseline: Logistic Regression with ALL features ===\n")
for s in seeds:
    set_seed(s)

    model_pkg_all = build_model_package(
        all_features_ind, feature_names, X_train_df, y_train_series, seed=s)

    X_test_scaled_all: numpy.ndarray = model_pkg_all["scaler"].transform(
        X_test[model_pkg_all["features"]].to_numpy())
    y_prob_all: numpy.ndarray = model_pkg_all["model"].predict_proba(X_test_scaled_all)[:, 1]

    test_auc_all: float = float(roc_auc_score(y_test, y_prob_all))
    test_ap_all: float = float(average_precision_score(y_test, y_prob_all))

    baseline_all_rows.append({
        "seed": s,
        "n_features": len(model_pkg_all["features"]),
        "test_auc": test_auc_all,
        "test_ap": test_ap_all,
    })
    print(f"  Seed {s} | Features: {len(model_pkg_all['features']):4d} "
          f"| Test AUC: {test_auc_all:.4f} | Test AP: {test_ap_all:.4f}")

baseline_all_df: pandas.DataFrame = pandas.DataFrame(baseline_all_rows)
print(f"\nMean test AUC: {baseline_all_df['test_auc'].mean():.4f} "
      f"(+/- {baseline_all_df['test_auc'].std():.4f})")
print(f"Mean test AP:  {baseline_all_df['test_ap'].mean():.4f} "
      f"(+/- {baseline_all_df['test_ap'].std():.4f})")

baseline_all_dir: str = os.path.join(RESULT_PATH_EVAL, "baseline_all_features")
_ensure_directory(baseline_all_dir)
baseline_all_df.to_csv(
    os.path.join(baseline_all_dir, "baseline_all_features.csv"), index=False)
print(f"\nBaseline (all features) results saved to: {baseline_all_dir}")

## Baseline: Logistic Regression with Forward Stepwise Feature Selection

Use scikit-learn's `SequentialFeatureSelector` (`direction="forward"`, 3-fold stratified CV on ROC AUC) to greedily add features while the cross-validated AUC keeps improving by more than `tol=1e-3`. Refit the final logistic regression on the selected subset via `build_model_package`, then report clean AUC/AP on the test set. This is the classical stepwise baseline the GA methods must justify improvement over.

In [ ]:
# Baseline 2: Logistic Regression with forward stepwise feature selection
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression as _SFSLogReg

baseline_fwd_rows: list[dict] = []

print("=== Baseline: Logistic Regression with Forward Stepwise Feature Selection ===\n")
for s in seeds:
    set_seed(s)

    # SFS needs scaled features; it refits LR internally at each step
    sfs_scaler: StandardScaler = StandardScaler()
    X_train_sfs: numpy.ndarray = sfs_scaler.fit_transform(X_train_df.to_numpy())

    sfs_cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=s)
    base_lr: _SFSLogReg = _SFSLogReg(
        penalty="l2", solver="lbfgs", max_iter=1000, random_state=s)
    sfs: SequentialFeatureSelector = SequentialFeatureSelector(
        base_lr,
        n_features_to_select="auto",
        tol=1e-3,
        direction="forward",
        scoring="roc_auc",
        cv=sfs_cv,
        n_jobs=-1,
    )
    sfs.fit(X_train_sfs, y_train_series)

    selected_mask: numpy.ndarray = sfs.get_support()
    fwd_individual: list[int] = [1 if m else 0 for m in selected_mask]

    # Re-fit the final LR on the selected subset through the standard helper
    model_pkg_fwd = build_model_package(
        fwd_individual, feature_names, X_train_df, y_train_series, seed=s)

    X_test_scaled_fwd: numpy.ndarray = model_pkg_fwd["scaler"].transform(
        X_test[model_pkg_fwd["features"]].to_numpy())
    y_prob_fwd: numpy.ndarray = model_pkg_fwd["model"].predict_proba(X_test_scaled_fwd)[:, 1]

    test_auc_fwd: float = float(roc_auc_score(y_test, y_prob_fwd))
    test_ap_fwd: float = float(average_precision_score(y_test, y_prob_fwd))

    baseline_fwd_rows.append({
        "seed": s,
        "n_features": len(model_pkg_fwd["features"]),
        "test_auc": test_auc_fwd,
        "test_ap": test_ap_fwd,
        "features": ",".join(model_pkg_fwd["features"]),
    })
    print(f"  Seed {s} | Features: {len(model_pkg_fwd['features']):4d} "
          f"| Test AUC: {test_auc_fwd:.4f} | Test AP: {test_ap_fwd:.4f}")

baseline_fwd_df: pandas.DataFrame = pandas.DataFrame(baseline_fwd_rows)
print(f"\nMean features: {baseline_fwd_df['n_features'].mean():.1f} "
      f"(+/- {baseline_fwd_df['n_features'].std():.1f})")
print(f"Mean test AUC: {baseline_fwd_df['test_auc'].mean():.4f} "
      f"(+/- {baseline_fwd_df['test_auc'].std():.4f})")
print(f"Mean test AP:  {baseline_fwd_df['test_ap'].mean():.4f} "
      f"(+/- {baseline_fwd_df['test_ap'].std():.4f})")

baseline_fwd_dir: str = os.path.join(RESULT_PATH_EVAL, "baseline_forward_stepwise")
_ensure_directory(baseline_fwd_dir)
baseline_fwd_df.to_csv(
    os.path.join(baseline_fwd_dir, "baseline_forward_stepwise.csv"), index=False)
print(f"\nBaseline (forward stepwise) results saved to: {baseline_fwd_dir}")